# Issue #272, part 4: ConvLSTM + a better loss, vs. the shipped C-collapse fix

**Status: not yet run.** This notebook is prepared and ready to go; training is
intentionally left un-executed until explicitly requested, since a full run here is
a multi-hour commitment (see the ETA cell below).

Companion to `issue_272_cnn_vit_cnn_compare.ipynb` (N-collapse vs. C-collapse vs. old
FIFO, 3 epochs, 6-day horizon -- a quick architectural sanity check) and
`issue_272_train_compare.ipynb` (toy model, real data). This notebook is a deliberately
longer, more useful run comparing two variables at once:

1. **Architecture**: `ConvLSTMProcessor` (new, this notebook) vs. `VitProcessor` with
   the shipped C-collapse rollout (PR #363, unmodified).
2. **Loss**: `WeightedBCEWithLogitsLoss` + ice-edge weighting (new) vs. plain `MSELoss`
   (what every existing comparison notebook has used so far).

**This is intentionally not a clean single-variable ablation** -- the ConvLSTM variant
gets both the new architecture *and* the new loss, while `c_collapse` is left exactly
as it's been trained in every prior notebook (MSE + sigmoid output). That's what was
asked for: "does the best candidate we can build beat the shipped fix", not "which of
these two changes is responsible". If the ConvLSTM variant wins, a follow-up ablation
(same loss on both, or same architecture with both losses) would be needed to attribute
the improvement. Worth keeping in mind when reading the result.

## What's different from the earlier notebook

- **Production hyperparameters, not the earlier notebook's stale ones.** PR #324
  (merged into `main` after the first comparison notebook was written) changed the real
  `cnn_vit_cnn.yaml` defaults substantially: latent space 144x144 -> 216x216, CNN
  encoder/decoder `n_layers` 3 -> 1, ViT `depth` 3 -> 6, `emb_dim` 128 -> 256, `heads`
  4 -> 8, `mlp_dim` 256 -> 1024, `patch_size` 16 -> 24. `c_collapse` here uses the
  *current* production config, not the earlier notebook's numbers.
- **Full 14-day forecast horizon** (`sic-icenet-14d`'s `n_forecast_steps=14`), not the
  6-day compromise used for the quick sanity check.
- **A real training budget**: `N_EPOCHS` below, not 3. See the ETA cell.
- **A bonus, unplanned speedup**: at 216x216 with `n_layers=1`, the CNN encoder/decoder
  no longer need the expensive resize step at all -- `216 * scale_factor^1 = 432`
  matches the native 432x432 SIC grid exactly, so there's no upsample-to-1152-then-
  downsample round trip like the old 144x144/`n_layers=3` config needed. Measured on
  this machine: ~0.6s/train-step for `c_collapse` at these settings (batch 8, 14-day
  horizon), vs. ~1.1-1.3s/step at the old 144x144/`n_layers=3` settings.

## The two variants

- **`c_collapse`**: real, unmodified `VitProcessor`, current production
  hyperparameters, `CNNDecoder(restrict_range="sigmoid")`, trained with plain `MSELoss`
  on the sigmoid-bounded `[0,1]` output. Exactly as used in every prior notebook.
- **`conv_lstm`**: new `ConvLSTMProcessor` (below), `hidden_channels=32` (chosen after
  timing a few options -- see the ETA cell), `CNNDecoder(restrict_range="none")` so the
  decoder returns raw logits, trained with `WeightedBCEWithLogitsLoss` and an ice-edge
  weighting (`4 * target * (1 - target)`, i.e. maximal weight where the target
  concentration is most ambiguous/near the ice edge, zero weight on the always-ice
  interior pack or always-open ocean far from it).

## Why this loss

Plain MSE on a sigmoid output has a real pathology for a field like SIC that's mostly
sharp 0s and 1s: MSE's gradient w.r.t. the pre-sigmoid logit is `2(p-t)*p*(1-p)`, and
the `p*(1-p)` factor vanishes as `p` saturates toward 0 or 1 -- exactly where most of
the grid sits, and exactly where persistence already scores well. `BCEWithLogitsLoss`
doesn't have this problem: its gradient w.r.t. the logit is just `(p-t)`, no vanishing
factor, so it can push the logit to the extremes needed to match sharp targets.
Treating SIC (a continuous area fraction) as a soft BCE target is a standard, legitimate
relaxation -- it's not being treated as a hard binary label.

**Important coupling**: `BCEWithLogitsLoss` expects raw logits and applies its own
internal sigmoid. That's why `conv_lstm`'s decoder uses `restrict_range="none"` --
using `sigmoid` there too would double-apply the sigmoid into the loss. `c_collapse`
keeps `restrict_range="sigmoid"` since it's paired with plain `MSELoss`, which expects
an already-bounded `[0,1]` prediction.

**Caveat**: neither variant uses the production `mask_type: active` land/never-ice
mask, since we don't have the generated mask files for this single-source ad hoc
setup (same caveat as every earlier notebook in this series).

In [ ]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # antialiased resize backward isn't
                                                   # implemented on MPS yet; fall back
                                                   # to CPU for that one op only. (Not
                                                   # actually exercised at n_layers=1,
                                                   # see markdown above, but harmless
                                                   # to leave set.)

import time

import numpy as np
import torch
from anemoi.datasets import open_dataset
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from icenet_mp.losses.weighted_bce_loss import WeightedBCEWithLogitsLoss
from icenet_mp.models.decoders.cnn_decoder import CNNDecoder
from icenet_mp.models.encoders.cnn_encoder import CNNEncoder
from icenet_mp.models.processors.base_processor import BaseProcessor
from icenet_mp.models.processors.vit import VitProcessor
from icenet_mp.types import DataSpace, ProcessorOutput

# Work around a fragile background telemetry pool in anemoi.datasets: open_dataset()
# unconditionally fires an analytics ping via a single-worker ProcessPoolExecutor
# created at import time. If that one worker process ever dies (plausible over a long
# Jupyter session with MPS activity/interrupted training runs), every subsequent
# open_dataset() call raises BrokenProcessPool, unrelated to our data/code. Analytics
# is already disabled locally (~/.config/anemoi/analytics.json) -- this just makes
# that opt-out crash-proof instead of a no-op that still touches the broken pool.
import anemoi.datasets.usage.analytics as _anemoi_analytics


class _NoOpExecutor:
    def submit(self, fn, *args, **kwargs):
        return None


_anemoi_analytics._executor = _NoOpExecutor()

torch.manual_seed(0)

ZARR = "/Volumes/Storage/ClimateData/data/anemoi/full-sicnorth-ssmis-25p0km-1979-2024-24h-v2.zarr"
N_HISTORY = 3
N_FORECAST = 14  # full production horizon (sic-icenet-14d), not the 6-day compromise
BATCH_SIZE = 8
N_EPOCHS = 15  # <-- the main knob: see the ETA cell just below for what this costs
LATENT_SPACE = (216, 216)  # current icenet_mp/config/model/cnn_vit_cnn.yaml default
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## ETA, measured on this machine (batch=8, 14-day horizon, production latent size)

**Note:** these numbers were measured against the OSI-SAF zarr (1992-2014 train
set, ~1049 steps/epoch). The data source has since been switched to SSMIS
(`sic-ssmis`, matching `predict=sic-ssmis-14d`), and the training window reverted to
1999-2014 (see the load-data cell below) since SSMIS's pre-1999 record has too many
missing dates for a clean contiguous window, unlike OSI-SAF's local 1992-2024 file.
That's ~730 steps/epoch, not ~1049 -- scaling the per-step timings below by
730/1049 gives roughly `c_collapse` ~7.3 min/epoch, `conv_lstm` ~12 min/epoch, i.e.
**roughly 5 hours total** at `N_EPOCHS=15`, not the ~7 hours below. This is an
adjustment of the earlier measurement, not a fresh benchmark on SSMIS -- re-time after
the first epoch or two of an actual SSMIS run if you need a firmer number.

| Variant | Measured train step | Steps/epoch (1992-2014 train set) | Time/epoch |
|---|---|---|---|
| `c_collapse` | ~0.6s | ~1049 | ~10.5 min |
| `conv_lstm` (hidden=32) | ~1.0s | ~1049 | ~17.5 min |

At `N_EPOCHS=15`: `c_collapse` ~157 min, `conv_lstm` ~262 min, trained sequentially ->
**roughly 7 hours total**, plus a few minutes for validation/test evaluation. Lower
`N_EPOCHS` above for a shorter run; both variants' cost scales linearly with it. (This
was measured at 1049 steps/epoch on the full 1992-2024 OSI-SAF dataset rather than
starting from 1999, which added 7 more years of training data and pushed steps/epoch
from ~730 to ~1049 -- see the note above on reverting to 1999 for SSMIS.)

`hidden_channels` was chosen empirically, not by default: a first attempt at
`hidden_channels=64` measured ~4s/step (~4x slower than `c_collapse`) -- a ConvLSTM's
gate convolutions run at full latent spatial resolution (216x216) on every one of
`n_history_steps + n_forecast_steps - 1` sequential steps, unlike a ViT, which reduces
to a small number of patch tokens (81, at `patch_size=24`) after one strided conv and
does its expensive (attention) work there instead. `hidden_channels=32` was the
best speed/capacity tradeoff found; `16` is faster still (~0.85s/step) if `N_EPOCHS`
needs to come down further.

**Progress while running**: the training loop below has a live `tqdm` progress bar
per epoch (updating every batch, with running loss in the postfix) plus an end-of-epoch
summary line, so you can watch it move in real time rather than staring at a silent
cell for 10-17 minutes. Validation and test-set evaluation loops also have bars, though
those pass much faster.

In [ ]:
print("Loading real SIC data from disk (one-time cost, ~1-2 minutes)...")
t0 = time.time()
# select=["ice_conc"]: unlike the OSI-SAF zarr this notebook used to load, the SSMIS
# zarr stores 6 variables (uncertainties, raw values, status flag, ice_conc); without
# an explicit select the reshape below would silently interleave them as if they were
# extra timesteps. fill_missing_dates="interpolate": SSMIS has one missing date inside
# this range (2000-12-01) -- without this, indexing across it raises MissingDateError.
ds = open_dataset(ZARR, select=["ice_conc"], fill_missing_dates="interpolate")
dates = list(ds.dates)
# The SSMIS zarr covers 1979-01-01 to 2024-12-31, but 1711 of its 1721 total missing
# dates fall before 1999 (very sparse coverage in the late 1970s/1980s) -- unlike the
# OSI-SAF zarr this notebook previously used, which was locally available from a clean
# 1992 start. So rather than "use all locally-available history", we start from 1999
# (matching float-argo's availability and the companion cnn_vit_cnn_compare notebook)
# to keep a clean, near-fully-populated window. Only one date is missing in
# 1999-2020 (2000-12-01), handled above via fill_missing_dates.
#
# SSMIS timestamps are at 12:00 (the OSI-SAF zarr this notebook used to load was at
# 00:00), so an exact `dates.index(np.datetime64("1999-01-01"))` lookup raises
# ValueError; match on day precision instead.
dates_by_day = np.array(dates, dtype="datetime64[D]")
idx_start = int(np.searchsorted(dates_by_day, np.datetime64("1999-01-01")))
idx_end = int(np.searchsorted(dates_by_day, np.datetime64("2020-12-31"), side="right") - 1)
raw = ds[idx_start : idx_end + 1].reshape(-1, 432, 432)  # (T, H, W)
sic = torch.from_numpy(np.ascontiguousarray(raw)).float()
split_dates = np.array(dates[idx_start : idx_end + 1])
print(f"Loaded {sic.shape[0]} days in {time.time() - t0:.1f}s, shape {tuple(sic.shape)}")

In [ ]:
# SSMIS timestamps are at 12:00, not midnight, so compare split boundaries by
# calendar day rather than exact instant (otherwise the day-of the boundary itself
# lands on the wrong side of the split).
split_dates_by_day = split_dates.astype("datetime64[D]")
train_end = np.searchsorted(split_dates_by_day, np.datetime64("2014-12-31"), side="right")
val_end = np.searchsorted(split_dates_by_day, np.datetime64("2017-12-31"), side="right")
print(f"train: {split_dates[0]} to {split_dates[train_end - 1]} ({train_end} days)")
print(f"validate: {split_dates[train_end]} to {split_dates[val_end - 1]} ({val_end - train_end} days)")
print(f"test: {split_dates[val_end]} to {split_dates[-1]} ({len(split_dates) - val_end} days)")


class WindowedSicDataset(Dataset):
    '''Slides (history, target) windows over an in-memory SIC tensor.'''

    def __init__(self, sic_thw: torch.Tensor, n_history: int, n_forecast: int) -> None:
        self.sic = sic_thw
        self.n_history = n_history
        self.n_forecast = n_forecast

    def __len__(self) -> int:
        return self.sic.shape[0] - self.n_history - self.n_forecast + 1

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        history = self.sic[idx : idx + self.n_history].unsqueeze(1)
        target = self.sic[
            idx + self.n_history : idx + self.n_history + self.n_forecast
        ].unsqueeze(1)
        return {"history": history, "target": target}


train_ds = WindowedSicDataset(sic[:train_end], N_HISTORY, N_FORECAST)
val_ds = WindowedSicDataset(sic[train_end:val_end], N_HISTORY, N_FORECAST)
test_ds = WindowedSicDataset(sic[val_end:], N_HISTORY, N_FORECAST)
print(f"train/val/test windows: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")

## The ConvLSTM processor

A `ConvLSTM` cell (Shi et al. 2015) is an LSTM with every gate's dense matmul replaced
by a convolution, so it keeps spatial structure while carrying a persistent
hidden/cell state through time -- the standard architecture for video-like
spatiotemporal sequence prediction (precipitation nowcasting, moving-MNIST-style
frame prediction, etc.).

Unlike `UNetProcessor`/`VitProcessor`, this doesn't fit the "concatenate a fixed window
of frames, call `forward` once" interface at all -- there's no fixed window; state
persists across arbitrarily many steps. So `ConvLSTMProcessor` doesn't use
`BaseProcessor.rollout` (the shipped C-collapse sliding window) or implement
`forward(x: TensorNCHW)` in the shared single-window sense -- it overrides `rollout`
completely, stepping through history frames to build up state, then continuing
autoregressively on its own predictions for the forecast horizon (no teacher forcing,
matching how every other processor in this codebase rolls out).

In [4]:
class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, kernel_size: int = 3) -> None:
        super().__init__()
        padding = kernel_size // 2
        self.hidden_channels = hidden_channels
        self.conv = nn.Conv2d(
            in_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size=kernel_size,
            padding=padding,
        )

    def forward(self, x: torch.Tensor, state: tuple[torch.Tensor, torch.Tensor]) -> tuple[torch.Tensor, torch.Tensor]:
        h_prev, c_prev = state
        gates = self.conv(torch.cat([x, h_prev], dim=1))
        i, f, o, g = torch.chunk(gates, 4, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c_prev + i * g
        h = o * torch.tanh(c)
        return h, c

    def init_state(
        self, batch_size: int, height: int, width: int, device: torch.device, dtype: torch.dtype
    ) -> tuple[torch.Tensor, torch.Tensor]:
        shape = (batch_size, self.hidden_channels, height, width)
        zeros = torch.zeros(shape, device=device, dtype=dtype)
        return (zeros, zeros.clone())


class ConvLSTMProcessor(BaseProcessor):
    '''Recurrent alternative to window-concatenation: state carries history forward
    instead of a fixed-size concatenated window. Overrides rollout() completely.'''

    def __init__(self, *, hidden_channels: int = 32, num_layers: int = 2, kernel_size: int = 3, **kwargs) -> None:
        super().__init__(**kwargs)
        in_channels = self.data_space.channels
        self.cells = nn.ModuleList(
            [
                ConvLSTMCell(
                    in_channels if i == 0 else hidden_channels, hidden_channels, kernel_size
                )
                for i in range(num_layers)
            ]
        )
        self.readout = nn.Conv2d(hidden_channels, self.data_space.channels, kernel_size=1)

    def _step(
        self, x_t: torch.Tensor, states: list[tuple[torch.Tensor, torch.Tensor]]
    ) -> tuple[torch.Tensor, list[tuple[torch.Tensor, torch.Tensor]]]:
        new_states, inp = [], x_t
        for cell, state in zip(self.cells, states, strict=True):
            h, c = cell(inp, state)
            new_states.append((h, c))
            inp = h
        return self.readout(inp), new_states

    def rollout(self, x: torch.Tensor, y: torch.Tensor | None = None) -> ProcessorOutput:  # noqa: ARG002
        batch, n_history, _, height, width = x.shape
        states = [
            cell.init_state(batch, height, width, x.device, x.dtype) for cell in self.cells
        ]
        # Warm up on the history window, building up recurrent state.
        last_pred = None
        for idx_t in range(n_history):
            last_pred, states = self._step(x[:, idx_t, :, :, :], states)
        # last_pred is already the forecast for day 1; continue autoregressively,
        # feeding each prediction back in as the next step's input (no teacher forcing,
        # matching every other processor's rollout in this codebase).
        outputs = [last_pred]
        current = last_pred
        for _ in range(self.n_forecast_steps - 1):
            current, states = self._step(current, states)
            outputs.append(current)
        return ProcessorOutput(prediction=torch.stack(outputs, dim=1))

## The loss: weighted BCE-with-logits + ice-edge weighting, for `conv_lstm` only

`c_collapse` keeps plain `MSELoss` (unweighted, on the sigmoid-bounded output) --
exactly as trained in every prior notebook, so it remains a like-for-like continuation
of that work.

In [5]:
bce_loss_fn = WeightedBCEWithLogitsLoss()


def edge_weight(target: torch.Tensor, floor: float = 0.05) -> torch.Tensor:
    '''Weight highest where the target concentration is most ambiguous (near the ice
    edge, target ~= 0.5) and lowest on the always-ice interior pack or always-open
    ocean (target ~= 0 or 1) -- exactly the cells persistence already gets right and
    that dominate a uniform per-pixel loss without teaching the model anything.'''
    return 4 * target * (1 - target) + floor


def c_collapse_loss(prediction: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return nn.functional.mse_loss(prediction, target)


def conv_lstm_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return bce_loss_fn(logits, target, edge_weight(target))

## Building both variants

Both share the same real `CNNEncoder`/`CNNDecoder` (production hyperparameters,
single `sic-ssmis` source -- matching `predict=sic-ssmis-14d`'s target, diverging
from earlier notebooks in this series which used `sic-icenet`/OSI-SAF), differing
only in processor, decoder `restrict_range`, and loss.

In [ ]:
DATA_SPACE = DataSpace(name="sic-ssmis", channels=1, shape=(432, 432))

VARIANTS = {
    "c_collapse": {
        "processor_cls": VitProcessor,
        "processor_kwargs": {
            "depth": 6,
            "dropout": 0.1,
            "emb_dim": 256,
            "heads": 8,
            "mlp_dim": 1024,
            "patch_size": 24,
        },
        "restrict_range": "sigmoid",
        "loss_fn": c_collapse_loss,
    },
    "conv_lstm": {
        "processor_cls": ConvLSTMProcessor,
        "processor_kwargs": {"hidden_channels": 32, "num_layers": 2, "kernel_size": 3},
        "restrict_range": "none",
        "loss_fn": conv_lstm_loss,
    },
}


def build_model(variant_name: str):
    spec = VARIANTS[variant_name]
    encoder = CNNEncoder(data_space_in=DATA_SPACE, latent_space=LATENT_SPACE, n_layers=1)
    combined_latent_space = DataSpace(
        name="combined_latent_space",
        channels=encoder.data_space_out.channels,
        shape=LATENT_SPACE,
    )
    processor = spec["processor_cls"](
        data_space=combined_latent_space,
        n_forecast_steps=N_FORECAST,
        n_history_steps=N_HISTORY,
        **spec["processor_kwargs"],
    )
    decoder = CNNDecoder(
        data_space_in=combined_latent_space,
        data_space_out=DATA_SPACE,
        n_layers=1,
        mask_type=None,
        restrict_range=spec["restrict_range"],
    )
    return encoder.to(DEVICE), processor.to(DEVICE), decoder.to(DEVICE)


def predict_prob(encoder, processor, decoder, variant_name: str, history: torch.Tensor) -> torch.Tensor:
    '''Both variants' raw model output, converted to a [0,1] concentration probability
    for evaluation/plotting -- c_collapse's decoder already outputs probabilities
    (restrict_range="sigmoid"); conv_lstm's decoder outputs raw logits that need an
    explicit sigmoid applied (kept separate from training, where the loss needs the
    raw logits for BCEWithLogitsLoss's own internal, numerically-stable sigmoid).'''
    latent_in = encoder.rollout(history)
    latent_out = processor.rollout(latent_in).prediction
    raw_output = decoder.rollout(latent_out)
    if VARIANTS[variant_name]["restrict_range"] == "none":
        return torch.sigmoid(raw_output)
    return raw_output


for name in VARIANTS:
    encoder, processor, decoder = build_model(name)
    n_params = sum(p.numel() for m in (encoder, processor, decoder) for p in m.parameters())
    print(f"{name}: {n_params:,} params")

## Training

Tracks both the native training loss (for monitoring each variant's own convergence
-- not comparable in value between the two, since MSE and weighted BCE are different
units) and a validation-set RMSE in probability space at the end of every epoch (fully
comparable between variants, via `predict_prob`), so there's an apples-to-apples signal
to plot even though the training losses themselves aren't on the same scale.

In [7]:
@torch.no_grad()
def val_rmse(encoder, processor, decoder, variant_name: str) -> float:
    encoder.eval(); processor.eval(); decoder.eval()
    loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    sq_err_sum, n_elements = 0.0, 0
    for batch in tqdm(loader, desc=f"  [{variant_name}] validating", leave=False):
        history = batch["history"].to(DEVICE)
        target = batch["target"].to(DEVICE)
        prediction = predict_prob(encoder, processor, decoder, variant_name, history)
        sq_err_sum += ((prediction - target) ** 2).sum().item()
        n_elements += target.numel()
    encoder.train(); processor.train(); decoder.train()
    return (sq_err_sum / n_elements) ** 0.5


def train(variant_name: str, n_epochs: int):
    encoder, processor, decoder = build_model(variant_name)
    loss_fn = VARIANTS[variant_name]["loss_fn"]
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    params = list(encoder.parameters()) + list(processor.parameters()) + list(decoder.parameters())
    optimizer = torch.optim.Adam(params, lr=1e-3)
    train_losses, val_rmses = [], []
    for epoch in range(n_epochs):
        epoch_loss, n_batches = 0.0, 0
        t0 = time.time()
        progress = tqdm(
            loader,
            desc=f"  [{variant_name}] epoch {epoch + 1}/{n_epochs}",
            leave=False,
        )
        for batch in progress:
            history = batch["history"].to(DEVICE)
            target = batch["target"].to(DEVICE)
            optimizer.zero_grad()
            latent_in = encoder.rollout(history)
            latent_out = processor.rollout(latent_in).prediction
            raw_output = decoder.rollout(latent_out)
            loss = loss_fn(raw_output, target)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
            progress.set_postfix(loss=f"{epoch_loss / n_batches:.5f}")
        train_losses.append(epoch_loss / n_batches)
        val_rmses.append(val_rmse(encoder, processor, decoder, variant_name))
        print(
            f"  [{variant_name}] epoch {epoch + 1}/{n_epochs}: "
            f"train loss = {train_losses[-1]:.5f}, val RMSE = {val_rmses[-1]:.5f}  "
            f"({time.time() - t0:.0f}s)"
        )
    return encoder, processor, decoder, train_losses, val_rmses


models = {}
train_loss_curves = {}
val_rmse_curves = {}
for name in VARIANTS:
    print(f"--- Training {name} ---")
    encoder, processor, decoder, train_losses, val_rmses = train(name, N_EPOCHS)
    models[name] = (encoder, processor, decoder)
    train_loss_curves[name] = train_losses
    val_rmse_curves[name] = val_rmses

--- Training c_collapse ---


  [c_collapse] epoch 1/15:   0%|          | 0/1049 [00:00<?, ?it/s]

  [c_collapse] validating:   0%|          | 0/135 [00:00<?, ?it/s]

  [c_collapse] epoch 1/15: train loss = 0.04574, val RMSE = 0.14348  (345s)


  [c_collapse] epoch 2/15:   0%|          | 0/1049 [00:00<?, ?it/s]

...

  [c_collapse] validating:   0%|          | 0/135 [00:00<?, ?it/s]

  [c_collapse] epoch 10/15: train loss = 0.00157, val RMSE = 0.04847  (447s)


  [c_collapse] epoch 11/15:   0%|          | 0/1049 [00:00<?, ?it/s]

Process SpawnProcess-1:
Traceback (most recent call last):
  File "/Users/aoife/.local/share/uv/python/cpython-3.12.11-macos-aarch64-none/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/aoife/.local/share/uv/python/cpython-3.12.11-macos-aarch64-none/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/aoife/.local/share/uv/python/cpython-3.12.11-macos-aarch64-none/lib/python3.12/concurrent/futures/process.py", line 252, in _process_worker
    call_item = call_queue.get(block=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/aoife/.local/share/uv/python/cpython-3.12.11-macos-aarch64-none/lib/python3.12/multiprocessing/queues.py", line 103, in get
    res = self._recv_bytes()
          ^^^^^^^^^^^^^^^^^^
  File "/Users/aoife/.local/share/uv/python/cpython-3.12.11-macos-aarch64-none/lib/python3.12/multiprocessing/connection.py", line 216, in recv_bytes
    b

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"c_collapse": "#1baf7a", "conv_lstm": "#3a7fd6"}
LABELS = {
    "c_collapse": "c_collapse (ViT, shipped fix, MSE)",
    "conv_lstm": "conv_lstm (weighted BCE + edge weighting)",
}
epochs = list(range(1, N_EPOCHS + 1))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name in VARIANTS:
    axes[0].plot(epochs, train_loss_curves[name], color=COLORS[name], marker="o", linewidth=2, label=LABELS[name])
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Train loss (native units -- not comparable across variants)")
axes[0].set_title("Training loss (own units per variant)")
axes[0].spines[["top", "right"]].set_visible(False)
axes[0].grid(axis="y", color="#e1e0d9", linewidth=1)
axes[0].legend(frameon=False, fontsize=8)

for name in VARIANTS:
    axes[1].plot(epochs, val_rmse_curves[name], color=COLORS[name], marker="s", linewidth=2, label=LABELS[name])
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation RMSE (probability space, comparable)")
axes[1].set_title("Validation RMSE (2015-2017, comparable across variants)")
axes[1].spines[["top", "right"]].set_visible(False)
axes[1].grid(axis="y", color="#e1e0d9", linewidth=1)
axes[1].legend(frameon=False, fontsize=8)

fig.tight_layout()
plt.show()

## Evaluation: per-lead-day skill on the held-out test set (2018-2020)

In [ ]:
@torch.no_grad()
def per_lead_day_rmse(predict_fn, dataset: WindowedSicDataset, variant_name: str, batch_size: int = 8) -> torch.Tensor:
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    sq_err_sum = torch.zeros(dataset.n_forecast)
    n_elements = 0
    for batch in tqdm(loader, desc=f"  evaluating {variant_name}", leave=False):
        history = batch["history"].to(DEVICE)
        target = batch["target"].to(DEVICE)
        prediction = predict_fn(history).cpu()
        target = target.cpu()
        sq_err = (prediction - target) ** 2
        sq_err_sum += sq_err.sum(dim=(0, 2, 3, 4))
        n_elements += sq_err.shape[0] * sq_err.shape[2] * sq_err.shape[3] * sq_err.shape[4]
    return torch.sqrt(sq_err_sum / n_elements)


def persistence_predict(history: torch.Tensor) -> torch.Tensor:
    return history[:, -1:, :, :, :].expand(-1, N_FORECAST, -1, -1, -1)


def model_predict_fn(variant_name: str):
    encoder, processor, decoder = models[variant_name]
    encoder.eval(); processor.eval(); decoder.eval()

    def predict(history: torch.Tensor) -> torch.Tensor:
        return predict_prob(encoder, processor, decoder, variant_name, history)

    return predict


rmse = {"persistence": per_lead_day_rmse(persistence_predict, test_ds, "persistence")}
for name in VARIANTS:
    rmse[name] = per_lead_day_rmse(model_predict_fn(name), test_ds, name)

print("Lead day:    " + "  ".join(f"{d:5d}" for d in range(1, N_FORECAST + 1)))
for name, values in rmse.items():
    print(f"{name:12s} " + "  ".join(f"{v:.3f}" for v in values.tolist()))
print()
for name, values in rmse.items():
    print(f"Mean RMSE over all {N_FORECAST} lead days -- {name}: {values.mean():.4f}")

In [ ]:
lead_days = list(range(1, N_FORECAST + 1))

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(lead_days, rmse["persistence"], color="#2a78d6", marker="o", markersize=5, linewidth=2, label="Persistence")
for name in VARIANTS:
    ax.plot(lead_days, rmse[name], color=COLORS[name], marker="s", markersize=5, linewidth=2, label=LABELS[name])
ax.set_xlabel("Forecast lead day")
ax.set_ylabel("RMSE (sea ice concentration)")
ax.set_title(f"Test-set forecast skill (2018-2020), {N_EPOCHS}-epoch run")
ax.set_xticks(lead_days)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e1e0d9", linewidth=1)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

## A qualitative look: one test forecast, three ways

In [ ]:
import cmocean

lead_day_to_show = N_FORECAST // 2
sample_idx = len(test_ds) // 2

sample = test_ds[sample_idx]
history = sample["history"].unsqueeze(0).to(DEVICE)
target = sample["target"].unsqueeze(0)

preds = {}
with torch.no_grad():
    for name in VARIANTS:
        preds[name] = model_predict_fn(name)(history).cpu()

truth_map = target[0, lead_day_to_show - 1, 0].numpy()
maps = {name: preds[name][0, lead_day_to_show - 1, 0].numpy() for name in VARIANTS}

fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
panels = [("truth", truth_map), *maps.items()]
titles = {"truth": f"Truth (day {lead_day_to_show})", **LABELS}
for ax, (name, arr) in zip(axes, panels, strict=True):
    im = ax.imshow(arr, cmap=cmocean.cm.ice, vmin=0, vmax=1)
    ax.set_title(titles[name], fontsize=9)
    ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.7, label="Sea ice concentration")
fig.suptitle("One test-set forecast, real SSMIS data", y=1.03)
plt.show()

## Summary

Fill in after running:

- Does `conv_lstm` (new architecture + new loss) beat `c_collapse` (shipped fix,
  unmodified loss) on validation RMSE and test-set RMSE, and at which lead days?
- Does either variant beat persistence at any lead day, and does the gap to
  persistence narrow with lead time (the pattern that would indicate the model is
  learning real dynamics rather than just a worse version of persistence)?
- Since architecture and loss changed together for `conv_lstm`, a real win here
  motivates a follow-up ablation (same loss on both variants, or `c_collapse` with the
  weighted BCE loss) to attribute the improvement -- not a conclusion from this run
  alone.